In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb

SEED = 42
PROCESSED_DATA_PATH = "../processed-data"
ORIGINAL_DATA_PATH = "../original-data"

In [2]:
print("1. Đọc dữ liệu Preprocessed")
df = pd.read_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Ép kiểu Category cho các cột chuỗi
object_cols = [col for col in df.select_dtypes(include=['object']).columns if col not in ['Date', 'Split']]
for col in object_cols:
    df[col] = df[col].astype('category')

print("2. Chuẩn bị tập Train toàn phần (2012-2022)...")
train_full = df[df['Split'] == 'Train'].copy()
test_data = df[df['Split'] == 'Test'].copy()

cols_to_drop = ['Date', 'Split', 'Revenue', 'COGS']
features = [col for col in df.columns if col not in cols_to_drop]

X_train_full = train_full[features]
y_train_rev = train_full['Revenue']
y_train_cogs = train_full['COGS']

# ==========================================
# 3. HUẤN LUYỆN LẠI TRÊN TOÀN BỘ DỮ LIỆU
# ==========================================
lgb_params = {
    'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.02, 
    'num_leaves': 127, 'n_estimators': 1500, 'random_state': SEED, 'n_jobs': -1
}

xgb_params = {
    'objective': 'reg:squarederror', 'learning_rate': 0.02, 
    'max_depth': 7, 'n_estimators': 1500, 'random_state': SEED, 'n_jobs': -1,
    'enable_categorical': True
}

print("-> Huấn luyện Ensemble cho Revenue...")
model_rev_lgb = lgb.LGBMRegressor(**lgb_params).fit(X_train_full, y_train_rev)
model_rev_xgb = xgb.XGBRegressor(**xgb_params).fit(X_train_full, y_train_rev)

print("-> Huấn luyện Ensemble cho COGS...")
model_cogs_lgb = lgb.LGBMRegressor(**lgb_params).fit(X_train_full, y_train_cogs)
model_cogs_xgb = xgb.XGBRegressor(**xgb_params).fit(X_train_full, y_train_cogs)

# ==========================================
# 4. (RECURSIVE)
# ==========================================
print("\n4. Bắt đầu dự báo đệ quy cho 548 ngày tập Test...")

# Sắp xếp tập test theo thời gian
test_data = test_data.sort_values('Date').reset_index(drop=True)
full_data_recursive = pd.concat([train_full, test_data]).sort_values('Date').reset_index(drop=True)

test_start_idx = full_data_recursive[full_data_recursive['Split'] == 'Test'].index[0]

for i in range(test_start_idx, len(full_data_recursive)):
    current_date = full_data_recursive.loc[i, 'Date']
    
    # CẬP NHẬT CÁC BIẾN LAGS TỪ KẾT QUẢ ĐÃ DỰ BÁO TRƯỚC ĐÓ
    for lag in [1, 7, 14, 30, 365]:
        full_data_recursive.loc[i, f'rev_lag_{lag}'] = full_data_recursive.loc[i-lag, 'Revenue']
        full_data_recursive.loc[i, f'cogs_lag_{lag}'] = full_data_recursive.loc[i-lag, 'COGS']
    
    # Lấy features của ngày hiện tại
    X_current = full_data_recursive.loc[[i], features]
    
    # Dự báo Revenue & COGS
    pred_rev = 0.5 * model_rev_lgb.predict(X_current)[0] + 0.5 * model_rev_xgb.predict(X_current)[0]
    pred_cogs = 0.5 * model_cogs_lgb.predict(X_current)[0] + 0.5 * model_cogs_xgb.predict(X_current)[0]
    
    # Lưu kết quả dự báo vào chính dataframe để ngày hôm sau sử dụng làm Lags
    full_data_recursive.loc[i, 'Revenue'] = max(0, pred_rev)
    full_data_recursive.loc[i, 'COGS'] = max(0, pred_cogs)
    
    if i % 100 == 0:
        print(f" Đã dự báo xong đến ngày: {current_date.date()}")

# ==========================================
# 5. XUẤT FILE SUBMISSION
# ==========================================
submission_final = full_data_recursive[full_data_recursive['Split'] == 'Test'][['Date', 'Revenue', 'COGS']]
submission_final['Date'] = submission_final['Date'].dt.strftime('%Y-%m-%d')
submission_final['Revenue'] = submission_final['Revenue'].round(2)
submission_final['COGS'] = submission_final['COGS'].round(2)

submission_final.to_csv('final_submission_ensemble_recursive.csv', index=False)

1. Đọc dữ liệu Preprocessed
2. Chuẩn bị tập Train toàn phần (2012-2022)...
-> Huấn luyện Ensemble cho Revenue...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000685 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2668
[LightGBM] [Info] Number of data points in the train set: 3468, number of used features: 26
[LightGBM] [Info] Start training from score 4243512.116196
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
-> Huấn luyện Ensemble cho COGS...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000576 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2668
[LightGBM